# Parse House Roll Call PDF files

This notebook parses and saves data to CSV files from the House Roll Call vote PDFs that are stored in ```../../data/HJRollCalls'``` subfolders.

## Parsing a Single PDF File

First, I will parse the first House Roll Call Vote pdf in our dataset (stored at ```../../data/HJRollCalls/2003/hj01082003.pdf```)

This approach uses PyMuPDF to extract text from the PDF by line, then applies regular expressions to appropriately capture the data. This method works well because we know that the structure of the House Roll Call Vote Journals is consistent across the years, so regular expressions can capture the names of the different representatives as long as they are robust enough to handle some varied formatting.

### Setup

In [125]:
import pymupdf
import re
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

Regular expressions needed to parse data are defined.

Depending on the year of the document, the required Regular expressions vary.

In [196]:
# Define regular expressions for validating a file's content
pre2009_patterns_to_match =[
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES\s+$',
    '^\s+Yea and Nay\s+(?P<RCDate>\d{2}/\d{2}/\d{2})\s(?P<RCTime>\d{2}:\d{2})\s\D$',
    '^\s+No.\s+(?P<RCNumberForYear>\d+)\s+(?P<YeaVoteCount>\d+)\s+YEAS\s+(?P<NayVoteCount>\d+)\s+NAYS\s+(?P<NotVotingCount>\d+)\s+N/V\s+$',
    '^\s+(?P<Vote1>[Y,N,X,P])\s+(?P<Name1>Mr. Speaker|(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote2>[Y,N,X,P])\s+(?P<Name2>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote3>[Y,N,X,P])\s+(?P<Name3>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+((?P<Vote4>[Y,N,X,P])\s+(?P<Name4>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+)?$',
    '^\s+\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING\s+$'
]

In [197]:
post2009pre2017_patterns_to_match = [
    '^YEAS: (?P<YeaVoteCount>\d+).?$',
    '^NAYS: (?P<NayVoteCount>\d+).?$',
    '^N-V: (?P<NVVoteCount>\d+).?$',
    '^P=PRESENT\s?$',
    '^X=NOT VOTING \s?$',
    '^\*=AFTER VOTE$',
    '^Yea and Nay Supplement$',
    '^JOURNAL OF THE HOUSE\.?$',
    '^(‘)?\D+,\s(?P<VoteDate>\D+\d{1,2},\s\d{4}).*$',
    '^Yea and Nay No\. (?P<RollCallCount>\d+).?$',
    '^(?P<VoteValue>[YNPXynpx]{1}).?$',
    '^(?P<Name>\*?([A-Za-z][A-ZÀ-ÿa-z.,\ \'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+).?$',
    '^YEAS: (?P<YeaVoteCount>\d+)\s+NAYS: (?P<NayVoteCount>\d+).?$',
    '^P=PRESENT        X=NOT VOTING      \*=AFTER VOTE$'
]

In [198]:
post2017_patterns_to_match = [
    '^(?P<RCDate>\d{2}/\d{2}/\d{4})\s(?P<RCTime>\d{2}:\d{2} \D{2})$',
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES$',
    '^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$',
    '^No.\s+(?P<RCNumberForYear>\d+)$',
    '^Yea and Nay$',
    '^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$',
    '^(?P<VoteValue>\D{1})$',
    '^\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING$',
    '^(?P<VoteValue>[YNPXynpx]{1})$',
    '^(?P<Name>Mr Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+).?$'
]

There are a few files that specifically do not match the patterns seen in many of the others. It appears that although the document appears the same way compared to files from after 2017, the files from 2023 and onwards fix some of the encoding issues that resulted in Nay votes appearing out of order and the main title and other features of the document not appearing in expected order.

While a more robust solution can be expanded (or split into a separate function) for simplicity for now, the functionality is being controlled by tracking the years to except and will be updated in the future during clean-up phase of this project

In [199]:
years_to_except = [2023, 2024, 2025]

In [293]:
year_to_court_number = {
    2003: 183,
    2004: 183,
    2005: 184,
    2006: 184,
    2007: 185,
    2008: 185,
    2009: 186,
    2010: 186,
    2011: 187,
    2012: 187,
    2013: 188,
    2014: 188,
    2015: 189,
    2016: 189,
    2017: 190,
    2018: 190,
    2019: 191,
    2020: 191,
    2021: 192,
    2022: 192,
    2023: 193,
    2024: 193,
    2025: 194,
}

A function for cleaning errors that were found through the OCR process. Some are specific to certain common cases, others are more general (removing special characters that should never be found in a name)

The removal of some character without replacing may cause issues when matching rep names later, but that is easier to deal with there than doing a unique case for each possible error.

In [294]:
def clean_ocr_name_errors(text):
    """
    Fixes common OCR errors in names:
    - '0' (zero) at start of word or after space → 'O' (letter O)
    - '1' (one) between letters → ''' (apostrophe)
    - '1' (one) at start of word → 'I' (letter I) - less common but possible
    
    Parameters:
    -----------
    text : str
        Text that may contain OCR errors
        
    Returns:
    --------
    str : Text with OCR errors corrected
    """
    if not text:
        return text
    
    # Replace '1' with apostrophe when it's between letters (like "L1Italien" → "L'Italien")
    # Pattern: letter, then 1, then letter
    text = re.sub(r'([A-Za-z])1([A-Za-z])', r"\1'\2", text)
    
    # Replace '0' with 'O' when it's at start of word or after space (like "0Flaherty" → "O'Flaherty")
    # But be careful - only if followed by a letter
    text = re.sub(r'(^|\s)0([A-Za-z\'])', r"\1O\2", text)
    
    # Also handle cases like "0 1 Flaherty" → "O'Flaherty"
    text = re.sub(r'0\s+1\s+', "O'", text)
    
    # Handle "L 1 Italien" → "L'Italien" (space between letter and 1)
    text = re.sub(r'([A-Za-z])\s+1\s+([A-Za-z])', r"\1'\2", text)

    # Handle *0-lrran -> *Curran
    text = re.sub(r'\*0-lrran', '*Curran', text)

    # Handle appearance of ◊ character
    text = re.sub(r'◊', 'O', text)

    # Handle 0111.inane
    text = re.sub(r'0111', r'Cull', text)

    # Handle Ca?rl)anale
    text = re.sub(r'Ca\?rl\)anale', r'Campanale', text)

    # Remove any remaining numeric characters (common OCR errors)
    # This catches any digits that weren't converted by the above rules
    text = re.sub(r'\d+', '', text)

    # remove invalid chars from strings except for important strings like * which mean something
    invalid_chars = r'[><|\\/\[\]{}`~@#$%^&+=_?;:"!]'
    text = re.sub(invalid_chars, '', text)

    # remove periods from name
    text = re.sub(r'\.', '', text)
    
    # Clean up any extra spaces that might have been left
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [295]:
clean_ocr_name_errors("0' Connell")

"O' Connell"

A helper function to help validate that the votes parsed match the values that are included with the sheet, for data verification confidence.

Interesting note:

For PDFs from 2017 or before, it appears that the House considered a present vote to be a N/V in the total sum. This can be checked by looking at Roll Call vote #100 for 2009 - where 2 N/Vs are recorded but there are no 'X' votes on the roll call, only 2 present votes. When tested with this condition, all votes (that parsed correctly and didn't drop votes due to failing to parse a rep's name) this held up until the year 2018.

Starting in 2018, present votes are not included with the sum of N/V. However, they are not included in any of the other categories either when looking at the totals given. To avoid an error in validation, I used the year to change the validation counting depending on this, but I also included a check for whether the total number of votes parsed matches the sum of the votes expected from the numbers.

In [296]:
def validate_vote_counts(vote_data, vote_counts, rc_year=2013, verbose=False, error_file=False):
    """
    Validates that the parsed vote counts match the actual votes.
    
    Parameters:
    -----------
    vote_data : list
        List of vote records, each as [court_number, rc_count_for_year, name, vote] where vote is 'Y', 'N', 'X', or 'P'
    vote_counts : list
        List of [yea_count, nay_count, nv_count] or [yea_count, nay_count, nv_count, ...]
    
    Returns:
    --------
    bool : True if counts match, False otherwise
    dict : Details about the validation (counts found, expected, differences)
    """

    if len(vote_counts) < 3:
        if verbose:
            print(f"Warning: vote_counts has fewer than 3 elements: {vote_counts}")
        return False, {"error": "vote_counts must have at least 3 elements"}
    
    expected_yea = vote_counts[0] if vote_counts[0] is not None else 0
    expected_nay = vote_counts[1] if vote_counts[1] is not None else 0
    expected_nv = vote_counts[2] if vote_counts[2] is not None else 0
    
    # Count actual votes
    actual_yea = sum(1 for vote in vote_data if vote[3].upper() == 'Y')
    actual_nay = sum(1 for vote in vote_data if vote[3].upper() == 'N')
    actual_nv = sum(1 for vote in vote_data if vote[3].upper() == 'X')
    actual_present = sum(1 for vote in vote_data if vote[3].upper() == 'P')
    
    # Check if counts match
    yea_match = actual_yea == expected_yea
    nay_match = actual_nay == expected_nay
    if rc_year < 2017: # Files from 2017 onward mostly consistently included present votes as N/V, with 1 single exception in 2017 and 1 exception in 2014
        nv_match = actual_nv + actual_present == expected_nv 
    else: 
        nv_match = actual_nv == expected_nv
    len_counts = actual_present + actual_nv + actual_nay + actual_yea
    len_expected = expected_nay + expected_yea + expected_nv
    len_match = len_counts == len_expected or len_expected < len_counts # when there are less expected votes than counted, tends to be an error to check
    
    all_match = yea_match and nay_match and nv_match
    
    validation_result = {
        "all_match": all_match,
        "yea": {
            "expected": expected_yea,
            "actual": actual_yea,
            "match": yea_match,
            "difference": actual_yea - expected_yea
        },
        "nay": {
            "expected": expected_nay,
            "actual": actual_nay,
            "match": nay_match,
            "difference": actual_nay - expected_nay
        },
        "nv": {
            "expected": expected_nv,
            "actual": actual_nv + actual_present,
            "match": nv_match,
            "difference": (actual_nv + actual_present) - expected_nv 
        },
        "present": {
            "count": actual_present
        },
        "total_votes_parsed": len(vote_data)
    }

    if error_file:
        print(f'Current file has known mistakes that have been manually validated, do not fail validation')
        all_match = True
    
    if verbose or not all_match:
        if not all_match:
            print(f"⚠️  Vote count validation FAILED:")
            print(f"   YEAS: Expected {expected_yea}, Found {actual_yea} (difference: {actual_yea - expected_yea})")
            print(f"   NAYS: Expected {expected_nay}, Found {actual_nay} (difference: {actual_nay - expected_nay})")
            if rc_year < 2014: print(f"   N-V:  Expected {expected_nv}, Found {actual_nv + actual_present} (difference: {(actual_nv + actual_present)- expected_nv})")
            else: print(f"   N-V:  Expected {expected_nv}, Found {actual_nv + actual_present} (difference: {(actual_nv + actual_present)- expected_nv})")
            print(f"   Present votes: {actual_present}")
            print(f"   Total votes parsed: {len(vote_data)}")
            print(f"   Total votes summed from document: {len_expected}")
            if len_match: print(f'   Validation failed but number of parsed votes matches so possible error on data part to check')
        else:
            print(f"✓ Vote count validation PASSED:")
            print(f"   YEAS: {actual_yea}, NAYS: {actual_nay}, N-V: {actual_nv}, Present: {actual_present}")
    
    return all_match, validation_result

A few files turned out to have incorrect information inputted by the MA Legislation House Journal, so those have been manually evaluated and fixes are saved here for later correction when those files appears

In [297]:
exception_files = [
    Path('..\..\data\HJRollCalls\\2009\\2009combined_Vote207.pdf'), # Incorrect yea and N/V, is 152 and 6, should be 150 and 8
    Path('..\..\data\HJRollCalls\\2009\\2009Combined_Vote250.pdf'), # Incorrect yea and N/V, is 152 and 7, should be 151 and 8
    Path('..\..\data\HJRollCalls\\2009\\2009Combined_Vote94.pdf'), # incorrect N/V, is 0, should be 2
    Path('..\..\data\HJRollCalls\\2010\\2010combined_Vote42.pdf'), # Incorrect Yea, is 358, should be 156
    Path('..\..\data\HJRollCalls\\2011\\2011combined_Vote103.pdf'), # Incorrect Yea and N/V, is 151 and 7, should be 153 and 5
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote2.pdf'), # incorrect N/V, is 6 should be 8
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote23.pdf'), # incorrect Nay and N/V is 5 and 2, should be 2 and 5
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote24.pdf'), # incorrect Nay and N/V, is 6 and 0, shoudl be 0 and 6
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote25.pdf'), # Incorrect Nay and N/V, is 5 and 0, should be 0 and 5
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote26.pdf'), # Incorrect Nay and N/V is 5 and 0, should be 0 and 5
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote27.pdf'), # Incorrect nay and N/V, is 5 and 0, should be 0 and 5
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote45.pdf'), # Incorrect yea, nay, and N/V, is 00, 0, and 0, should be 37, 118, 3
    Path('..\..\data\HJRollCalls\\2013\\2013combined_Vote150.pdf'), # Incorrect N/V, is 0, should be 3
    Path('..\..\data\HJRollCalls\\2014\\2014combined_Vote202.pdf'), # Incorrect N/V says 3, should be 5 - Note: this file is an exception to the rule for pre-2017 files so I am treating it as a mistake in this case. 
    Path('..\..\data\HJRollCalls\\2016\\2016combined_Vote117.pdf'), # Incorrect Yea and N/V, is 153 and 7, should be 152 and 8
    Path('..\..\data\HJRollCalls\\2016\\2016combined_Vote81.pdf'), # Incorrect Nay and N/V, is 110 and 5, should be 109 and 6
    Path('..\..\data\HJRollCalls\\2017\\2017combined_Vote241.pdf'), # Incorrect Yea Nay and N/V, is 125, 30, and 3, should be 143, 11, 4
    Path('..\..\data\HJRollCalls\\2017\\2017combined_Vote46.pdf'), # Incorrect N-V, is 3, should be 2. **This one is odd becuase files at this point did not count present as N/V so this is operating under that same assumption.
    Path('..\..\data\HJRollCalls\\2017\\2017combined_Vote54.pdf'), # Incorrect Nay and N/V, is 0 and 1, should be 1 and 0
]

exception_corrections = {
    Path('..\..\data\HJRollCalls\\2009\\2009combined_Vote207.pdf'): [150, 2, 8],
    Path('..\..\data\HJRollCalls\\2009\\2009Combined_Vote250.pdf'): [151, 1, 8],
    Path('..\..\data\HJRollCalls\\2009\\2009Combined_Vote94.pdf'): [50, 107, 2],
    Path('..\..\data\HJRollCalls\\2010\\2010combined_Vote42.pdf'): [156, 0, 3],
    Path('..\..\data\HJRollCalls\\2011\\2011combined_Vote103.pdf'): [153, 0, 5],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote2.pdf'): [152, 0, 8],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote23.pdf'): [151, 2, 5],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote24.pdf'): [152, 0, 6],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote25.pdf'): [153, 0, 5],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote26.pdf'): [153, 0, 5],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote27.pdf'): [153, 0, 5],
    Path('..\..\data\HJRollCalls\\2012\\2012combined_Vote45.pdf'): [37, 118, 3],
    Path('..\..\data\HJRollCalls\\2013\\2013combined_Vote150.pdf'): [38, 115, 3],
    Path('..\..\data\HJRollCalls\\2014\\2014combined_Vote202.pdf'): [150, 0, 5],
    Path('..\..\data\HJRollCalls\\2016\\2016combined_Vote117.pdf'): [152, 0, 8],
    Path('..\..\data\HJRollCalls\\2016\\2016combined_Vote81.pdf'): [45, 109, 6],
    Path('..\..\data\HJRollCalls\\2017\\2017combined_Vote241.pdf'): [143, 11, 4],
    Path('..\..\data\HJRollCalls\\2017\\2017combined_Vote46.pdf'): [157, 0, 2],
    Path('..\..\data\HJRollCalls\\2017\\2017combined_Vote54.pdf'): [159, 1, 0],
}

Parsing functions for the different years of PDFs are defined. Currently PDFs from before 2009 and after 2017 are supported and are defined in the following two functions.

In [298]:
def parse_Pre2009_PDF(path, bill_id, court_number, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(court_number)

        rc_count_for_year = 0

        current_pdf_vote_data = []

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 2: # parse full title line individually for ease
                current_pdf_bill_data.append(line.strip())

            # match = re.search(title_regex, line)
            matched = False
            for j, pattern in enumerate(pre2009_patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    if j == 0:
                        # Do nothing, just the title
                        if verbose: print("Parsing title line")
                    elif j == 1:
                        # Extract vote date and time
                        rc_date = match.group('RCDate')
                        rc_date = datetime.strptime(rc_date, "%m/%d/%y").strftime("%m/%d/%Y")
                        current_pdf_bill_data.append(rc_date)
                        rc_time = match.group('RCTime')
                        if int(rc_time[:2]) < 12 and int(rc_time[:2]) >= 11:
                            rc_time += ' AM'
                        else: 
                            rc_time += ' PM'
                        current_pdf_bill_data.append(rc_time)
                        if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                    elif j == 2:
                        rc_count_for_year = match.group('RCNumberForYear')
                        current_pdf_bill_data.append(int(rc_count_for_year))

                        rc_yea_vote_count = match.group('YeaVoteCount')
                        current_pdf_bill_data.append(int(rc_yea_vote_count))

                        rc_nay_vote_count = match.group('NayVoteCount')
                        current_pdf_bill_data.append(int(rc_nay_vote_count))

                        rc_not_voting_count = match.group('NotVotingCount')
                        current_pdf_bill_data.append(int(rc_not_voting_count))

                        if verbose: print(f'Found {rc_yea_vote_count} YEA votes, {rc_nay_vote_count} Nay votes, {rc_not_voting_count} NV votes for RC Vote {rc_count_for_year}')
                    elif j == 3:
                        vote1 = match.group('Vote1')
                        name1 = match.group('Name1')
                        current_pdf_vote_data.append([court_number, int(rc_count_for_year), name1, vote1])

                        vote2 = match.group('Vote2')
                        name2 = match.group('Name2')
                        current_pdf_vote_data.append([court_number, int(rc_count_for_year), name2, vote2])

                        vote3 = match.group('Vote3')
                        name3 = match.group('Name3')
                        current_pdf_vote_data.append([court_number, int(rc_count_for_year), name3, vote3])

                        vote4 = match.group('Vote4')
                        name4 = match.group('Name4')
                        if not vote4 == None: current_pdf_vote_data.append([court_number, int(rc_count_for_year), name4, vote4])

                        if verbose: print(f'Found {name1} with vote {vote1}, {name2} with vote {vote2}, {name3} with {vote3}')
                    elif j == 4:
                        if verbose: print("Parsing end line")

            if not matched:
                Exception(f"Failed to match regex on line {i} : {line}")

        
        # Get vote counts from current_pdf_bill_data list (indicex)
        vote_counts = current_pdf_bill_data[6:9]
        votes_match, results = validate_vote_counts(current_pdf_vote_data, vote_counts)
        current_pdf_bill_data.append(votes_match)
        

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

In [299]:
def parse_2009through2017_PDF(path, bill_id, court_number, file_year, verbose=False):
    """
    Clean rewrite of the parser with simpler logic and better bill name handling.
    Structure:
    1. Parse metadata (vote counts, date, roll call number)
    2. Parse bill name (collect lines until we hit a vote)
    3. Parse votes (vote + name pairs)
    """
    import sys
    
    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page in doc:
        text = page.get_text()
        lines = [line.strip() for line in text.split('\n') if line.strip()]  # Remove blank lines
        
        # Initialize data structures
        bill_data = [bill_id, court_number]
        vote_data = []
        rc_count_for_year = 0
        
        # State tracking
        metadata_parsed = False
        bill_name_lines = []
        in_bill_name = False
        bill_name_complete = False
        skip_next = False  # Skip next line if it was already processed as a name
        parsed_double = False
        
        # Step 1: Parse metadata and find bill name start
        for i, line in enumerate(lines):
            # Skip line if it was already processed (as a name following a vote)
            if skip_next:
                skip_next = False
                continue
            # Parse combined YEAS/NAYS line
            match = re.match(r'^YEAS:\s*(?P<YeaVoteCount>\d+)\s+NAYS:\s*(?P<NayVoteCount>\d+)', line)
            if match:
                bill_data.append(int(match.group('YeaVoteCount')))
                bill_data.append(int(match.group('NayVoteCount')))
                parsed_double = True
                continue
            # Parse YEAS count
            match = re.match(r'^YEAS:\s*(?P<YeaVoteCount>\d+)', line)
            if match and not parsed_double:
                bill_data.append(int(match.group('YeaVoteCount')))
                continue
            
            # Parse NAYS count
            match = re.match(r'^NAYS:\s*(?P<NayVoteCount>\d+)', line)
            if match:
                bill_data.append(int(match.group('NayVoteCount')))
                continue
            
            # Parse N-V count
            match = re.match(r'^N-V:\s*(?P<NVVoteCount>\d+)', line)
            if match:
                bill_data.append(int(match.group('NVVoteCount')))
                continue
               
            # Parse date
            match = re.match(r'.*,\s*(?P<VoteDate>\w+\s+\d{1,2},\s+\d{4})', line)
            if match:
                date_string = match.group('VoteDate')
                if 'Ocotober' in date_string:
                    date_string = date_string.replace('Ocotober', 'October')
                try:
                    datetime_object = datetime.strptime(date_string, "%B %d, %Y")
                    bill_data.append(datetime_object.strftime("%m/%d/%Y"))
                except:
                    bill_data.append('01/01/1900')
                bill_data.append(None)  # No time for these years
                continue
            
            # Parse roll call number
            match = re.match(r'^Yea and Nay No\.\s*(?P<RollCallCount>\d+)', line)
            if match:
                rc_count_for_year = int(match.group('RollCallCount'))
                bill_data.append(int(rc_count_for_year))
                metadata_parsed = True
                in_bill_name = True  # Next lines are bill name
                continue
            
            # Step 2: Collect bill name lines (after roll call number, before votes)
            if in_bill_name and not bill_name_complete:
                # Check if this line is a vote value (Y, N, X, P)
                vote_match = re.match(r'^(?P<VoteValue>[YNPXynpx]{1})$', line)
                if vote_match:
                    # We've hit the first vote, bill name is complete
                    bill_name_complete = True
                    in_bill_name = False
                    # Process this vote
                    vote = vote_match.group('VoteValue').upper()
                    if i + 1 < len(lines):
                        name_line = lines[i + 1]
                        name_line = clean_ocr_name_errors(name_line)
                        name_match = re.match(post2009pre2017_patterns_to_match[11], name_line)
                        if name_match:
                            name = name_match.group('Name').replace('.', '').strip()
                            vote_data.append([court_number, int(rc_count_for_year), name, vote])
                            skip_next = True  # Skip the name line on next iteration
                else:
                    # This is part of the bill name
                    bill_name_lines.append(line)
                continue
            
            # Step 3: Parse votes (vote + name pairs)
            if bill_name_complete:
                # Check if this is a vote value
                vote_match = re.match(r'^(?P<VoteValue>[YNPXynpx]{1})$', line)
                if vote_match:
                    vote = vote_match.group('VoteValue').upper()
                    # Next line should be the name
                    if i + 1 < len(lines):
                        name_line = lines[i + 1]
                        if name_line == '0':
                            name_line += ' '
                            name_line += lines[i + 2] # Handle occasional O'Day/O'Flaherty edge case
                        name_line = clean_ocr_name_errors(name_line)
                        name_match = re.match(post2009pre2017_patterns_to_match[11], name_line)
                        if name_match:
                            name = name_match.group('Name').replace('.', '').strip()
                            vote_data.append([court_number, int(rc_count_for_year), name, vote])
                            skip_next = True  # Skip the name line on next iteration
                    continue
                
                # Check if this is a name (might be standalone if vote was on previous line)
                name_match = re.match(post2009pre2017_patterns_to_match[11], line)
                if name_match:
                    # Check if previous line was a vote
                    if i > 0:
                        prev_vote_match = re.match(r'^(?P<VoteValue>[YNPXynpx]{1})$', lines[i-1])
                        if prev_vote_match:
                            vote = prev_vote_match.group('VoteValue').upper()
                            name = name_match.group('Name').replaceall('.', '').strip()
                            vote_data.append([court_number, int(rc_count_for_year), name, vote])
                        # Check if this is the odd case where the vote appears on the same line X Atsalis
                        same_line_match = re.match(r'^[XPYN]{1}\s(?P<Name>\*?([A-Za-z][A-ZÀ-ÿa-z.,\ \'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+).?$', line)
                        if same_line_match:
                            print("Handling special vote case for line: ", line)
                            split_line = line.split(' ')
                            vote_data.append([court_number, int(rc_count_for_year), split_line[1], split_line[0]])

                    continue
      
        # Combine bill name lines
        bill_name = ' '.join(bill_name_lines) if bill_name_lines else ''
        
        # Build final bill_data in correct order
        # Expected: [bill_id, court_number, bill_name, rollcall_date, rollcall_time, 
        #            rollcall_count_for_year, yea_vote_count, nay_vote_count, not_voting_count]
        
        # Extract values from bill_data (which currently has: [bill_id, court_number, yea?, nay?, nv?, date?, time?, rc_count?])
        final_bill_data = [bill_id, court_number, bill_name]
        
        # Find date (string with /)
        date_val = next((x for x in bill_data[2:] if x and isinstance(x, str) and '/' in str(x) and len(str(x)) == 10), None)
        if date_val is None:
            bill_data.insert(5, '01/01/1900')
            bill_data.insert(6, None) # for the time
        final_bill_data.append(date_val if date_val else '01/01/1900')
        
        # Time is always None for 2009-2017
        final_bill_data.append(None)
        
        # Roll call count
        rc_val = bill_data[7]
        final_bill_data.append(int(rc_val) if rc_val else None)
        
        # Vote counts 
        vote_counts = bill_data[2:5]
        
        file_has_error = False
        if exception_corrections.get(Path(path)) is not None:
            print(f'Matched path, except this file')
            file_has_error = True

        votes_match, results = validate_vote_counts(vote_data, vote_counts, rc_year=file_year, error_file=file_has_error)

        # If this is a known file with errors, get the corrected values for votes.
        if file_has_error:
            print('getting fixed rc vote counts')
            vote_counts = exception_corrections.get(Path(path))
            


        # Order: yea, nay, n/v
        final_bill_data.append(vote_counts[0] if len(vote_counts) > 0 else None)
        final_bill_data.append(vote_counts[1] if len(vote_counts) > 1 else None)
        final_bill_data.append(vote_counts[2] if len(vote_counts) > 2 else None)
        
        bill_data = final_bill_data
        bill_data.append(votes_match)
        
        all_pdf_bills.append(bill_data)
        all_pdf_votes.extend(vote_data)
        bill_id += 1

    return all_pdf_bills, all_pdf_votes, bill_id

In [300]:
def parse_Post2017_PDF(path, bill_id, court_number, rc_year, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    file_year = int(str(path)[23:27])

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(int(court_number))
        rc_count_for_year = 0

        current_pdf_vote_data = []
        nay_vote_found = False
        end_found = False
        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 7 and file_year not in years_to_except: # parse full title line individually for ease
                current_pdf_bill_data.insert(2, line.strip())
            elif file_year in years_to_except and i == 1:
                current_pdf_bill_data.insert(2, line.strip())

            # match = re.search(title_regex, line)
            matched = False
            if not end_found:
                for j, pattern in enumerate(post2017_patterns_to_match):
                    match = re.match(pattern, line)

                    if match:
                        matched = True
                        if verbose: print(f'Matched with pattern {pattern}')

                        if j == 0:
                            # Extract vote date and time
                            rc_date = match.group('RCDate')
                            current_pdf_bill_data.append(rc_date)
                            rc_time = match.group('RCTime')
                            current_pdf_bill_data.append(rc_time)
                            if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                        elif j == 1:
                            # Do nothing, just the title
                            if verbose: print("Parsing title line")
                            break
                        elif j == 2:
                            # Get vote count and type of vote since they appear each on a new line
                            rc_vote_count = match.group('VoteCount')
                            current_pdf_bill_data.append(int(rc_vote_count))

                            rc_vote_type = match.group('VoteType')

                            if verbose: print(f'Found {rc_vote_count} {rc_vote_type} votes')
                        elif j == 3:
                            # Parse RC Vote number
                            rc_count_for_year = match.group('RCNumberForYear')
                            current_pdf_bill_data.insert(3, int(rc_count_for_year)) # Insert at index so it matches format of other PDF styles


                            if verbose: print(f'Found No. {rc_count_for_year} on current line')
                        elif j == 4:
                            # Parsing Yays and Nays text, do nothing
                            if verbose: print("Parsing \'Yays and Nays\'")
                            break
                        elif j == 5:
                            # Skip parsing this line if previous line was a Nay Vote
                            if nay_vote_found and file_year not in years_to_except:
                                nay_vote_found = False
                                continue

                            # Parsing Voter Names
                            name = match.group('Name')

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i+1])
                            vote = vote_match.group('VoteValue')

                            current_pdf_vote_data.append([court_number, int(rc_count_for_year), name, vote])

                            if verbose: print(f'Found vote {vote} from Rep. {name}')
                        elif j == 6:
                            # Parse vote value, if value is N for Nay, parse name as well since they appear in reverse order

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i])
                            vote = vote_match.group('VoteValue')

                            if vote == 'N' and file_year not in years_to_except:
                                name_match = re.match('^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$', lines[i+1])
                                name = name_match.group('Name')

                                current_pdf_vote_data.append([court_number, int(rc_count_for_year), name, vote])
                                nay_vote_found = True
                                if verbose: print("Found Nay vote, will skip parsing next name")
                            else:
                                continue
                        elif j == 7:
                            if verbose: print("Parsing end line")
                            end_found = True
                            break
                if not matched:
                    Exception(f"Failed to match regex on line {i} : {line}")

        

        # Swap again for years in years to except
        if rc_year in years_to_except:
            current_pdf_bill_data[3], current_pdf_bill_data[5] = current_pdf_bill_data[5], current_pdf_bill_data[3]
            current_pdf_bill_data[3], current_pdf_bill_data[4] = current_pdf_bill_data[4], current_pdf_bill_data[3]
        else:
            current_pdf_bill_data[4], current_pdf_bill_data[5] = current_pdf_bill_data[5], current_pdf_bill_data[4]

        # Get vote counts from current_pdf_bill_data list (indicex)
        vote_counts = current_pdf_bill_data[6:9]
        # validate vote counts and append result to array
        votes_match, results = validate_vote_counts(current_pdf_vote_data, vote_counts, rc_year=rc_year)
        current_pdf_bill_data.append(votes_match)

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

### Testing on a Single PDF document

In [237]:
bill_data = []
vote_data = []
bill_id = 0
path = "../../data/HJRollCalls/2003/hj01082003.pdf"

# Call defined function to get data for current pdf
pdf_bill_data, pdf_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id, 182)

bill_data.extend(pdf_bill_data)
vote_data.extend(pdf_vote_data)

2013
2013
2013
2013
2013
2013
Succesfully parsed file at path: ../../data/HJRollCalls/2003/hj01082003.pdf


In [238]:
bill_data

[[0, 182, 'QUORUM', '01/08/2003', '01:44 PM', 1, 151, 0, 9, True],
 [1,
  182,
  'H2003  On motion to       Postpone',
  '01/08/2003',
  '02:01 PM',
  2,
  45,
  107,
  8,
  True],
 [2,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '02:20 PM',
  3,
  41,
  110,
  9,
  True],
 [3,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '02:45 PM',
  4,
  44,
  107,
  9,
  True],
 [4,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '04:03 PM',
  5,
  28,
  119,
  13,
  True],
 [5,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '04:34 PM',
  6,
  22,
  128,
  10,
  True]]

In [57]:
vote_data

[[182, '1', 'Mr. Speaker', 'Y'],
 [182, '1', 'Fresolo', 'Y'],
 [182, '1', 'Murphy, K.J.', 'Y'],
 [182, '1', 'Swan', 'Y'],
 [182, '1', 'Asselin', 'Y'],
 [182, '1', 'Galvin', 'Y'],
 [182, '1', 'Nangle', 'Y'],
 [182, '1', 'Teahan', 'Y'],
 [182, '1', 'Atkins', 'Y'],
 [182, '1', 'Garry', 'Y'],
 [182, '1', 'Naughton', 'Y'],
 [182, '1', 'Timilty', 'Y'],
 [182, '1', 'Atsalis', 'Y'],
 [182, '1', 'Gobi', 'X'],
 [182, '1', 'Nyman', 'Y'],
 [182, '1', 'Tobin', 'Y'],
 [182, '1', 'Ayers', 'Y'],
 [182, '1', 'Goguen', 'Y'],
 [182, '1', "O'Brien", 'Y'],
 [182, '1', 'Toomey', 'Y'],
 [182, '1', 'Balser', 'Y'],
 [182, '1', 'Golden B.P.', 'Y'],
 [182, '1', "O'Flaherty", 'Y'],
 [182, '1', 'Torrisi', 'Y'],
 [182, '1', 'Binienda', 'Y'],
 [182, '1', 'Golden T.A.', 'Y'],
 [182, '1', 'Owens-Hicks', 'Y'],
 [182, '1', 'Travis', 'Y'],
 [182, '1', 'Blumer', 'Y'],
 [182, '1', 'Grant', 'Y'],
 [182, '1', 'Parente', 'Y'],
 [182, '1', 'Turkington', 'Y'],
 [182, '1', 'Bosley', 'Y'],
 [182, '1', 'Greene', 'Y'],
 [182, '1', 

In [11]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,,
0,185,On adoption of an amendment (offered by Mr. Jo...,01/14/2009,None,2,18,137,5


In [12]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['court_number', 'rollcall_count_for_year', 'voter_name', 'vote_value'])
vote_df

,court_number,rollcall_count_for_year,voter_name,vote_value
0,185,2,Mr Speaker,N
1,185,2,Ehrlich,N
2,185,2,Madden,N
3,185,2,Speliotis,N
4,185,2,Aguiar,N
...,...,...,...,...
155,185,2,Smola,Y
156,185,2,Dykema,N
157,185,2,L Italien,N
158,185,2,Smizik,N


## Batch Processing all PDFs

Now that we have a function for scraping the data from all pages of a single pdf, we can loop through all of the pdfs that are stored on disk in the data and create the full dataset

In [301]:
# Get path to all pdfs stored in HJRollCall folder
all_paths = []
root_path = Path('../../data/HJRollCalls')

for path in root_path.glob('**/*.pdf'):
    all_paths.append(path)

print(len(all_paths))

2509


**Put file '..\..\data\HJRollCalls\2016\2016combined_Vote98.pdf' on a watch list, does not parse at all**



**file '..\..\data\HJRollCalls\2017\2017combined_Vote233.pdf' has a very odd bug - Goldstein-rose * is eating the next vote and vanishing it, I may just fix it manually for now**

**file '..\..\data\HJRollCalls\2017\2017combined_Vote40.pdf' same bug as above, too long name, eating a vote. will manually fix for now, then deal with later**


### Parse all PDFs

This block is the core part of this notebook that actually iterates over the whole folder of PDFs and parses them.

Currently, this function is set to parse roll call votes that are either < 2009 or 2017 >. This is because PDFs from the years 2009 - 2017 are done is a specific style where the actual votes are stored as an image embedded in the PDF, not text, and require some special attention. In the worst case, this data may need to be manually scraped but future experiementation will be done to see if we can automate this process.

Each parse_####_PDF() function call takes 3 variables, path = the file path to parse, bill_id = the iterator of all bills parsed (used for reference in the database table), and an optional named parameter verbose, which is set to False by default. When verbose is set to True, the function will output debugging messages in case something goes wrong during the parsing process.

In [312]:
bill_id = -1
bill_data = []
vote_data = []

for path in tqdm(all_paths):

    # If path includes the year <2009, skip for now
    file_year = int(str(path)[23:27])
    print(file_year)
    court_number = year_to_court_number[file_year]
    print(court_number)

    print(f'Attempting to parsing file at path: {path}')
    if file_year < 2009:
        if bill_id < 0: bill_id+1
        current_bill_data, current_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id+1, court_number)
    # Skip files from 2009 - 2017 for now since they will have more things to deal with for parsing
    elif file_year >= 2009 and file_year <= 2017:
        if bill_id < 0: bill_id+1
        current_bill_data, current_vote_data, bill_id = parse_2009through2017_PDF(path, bill_id+1, court_number, file_year)
    elif file_year > 2017:
        if bill_id < 0: bill_id+1
        current_bill_data, current_vote_data, bill_id = parse_Post2017_PDF(path, bill_id+1, court_number, file_year)
    else:
        print(f"File in unexpected year attempted to be parsed: {path, file_year}")
        break

    bill_data.extend(current_bill_data)
    vote_data.extend(current_vote_data)

  0%|          | 0/2509 [00:00<?, ?it/s]

2003
183
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01082003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01082003.pdf
2003
183
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01092003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01092003.pdf
2003
183
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01142003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01142003.pdf
2003
183
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01222003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01222003.pdf
2003
183
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj02032003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj02032003.pdf
2003
183
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj02052003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj02052003.pdf
2003
183
Attempt

### Creating and writing DataFrames from parsed data then converting to CSV

Get the bill data and output into a dataframe

In [345]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count', 'validated'])

bill_df.set_index('bill_id', inplace=True)
bill_df.rollcall_count_for_year = bill_df.rollcall_count_for_year.astype('int64')
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count,validated
bill_id,,,,,,,,,
0,183,QUORUM,01/08/2003,01:44 PM,1,151,0.0,9.0,True
1,183,H2003 On motion to Postpone,01/08/2003,02:01 PM,2,45,107.0,8.0,True
2,183,H2003 On Amendment,01/08/2003,02:20 PM,3,41,110.0,9.0,True
3,183,H2003 On Amendment,01/08/2003,02:45 PM,4,44,107.0,9.0,True
4,183,H2003 On Amendment,01/08/2003,04:03 PM,5,28,119.0,13.0,True
...,...,...,...,...,...,...,...,...,...
7962,194,H. 4240 Shall item 7004-0109 (Shelter Workforc...,10/08/2025,02:13 PM,84,131,24.0,5.0,True
7963,194,H. 4240 Shall item 7004-3036 (Housing Services...,10/08/2025,02:17 PM,85,131,24.0,5.0,True
7964,194,H. 4240 Shall item 7007-0150 (Regional Economi...,10/08/2025,02:23 PM,86,153,2.0,5.0,True


Check if any bills failed to validate the votes

In [352]:
bill_df[bill_df['validated'] == False]

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count,validated
bill_id,,,,,,,,,


Get the vote data and write to dataframe

In [348]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['court_number', 'rollcall_count_for_year', 'voter_name', 'vote_value'])
vote_df.rollcall_count_for_year = vote_df.rollcall_count_for_year.astype('int64')

The parsed output can be written to csv files for future cleaning and processing

In [350]:
bill_df.to_csv('../../data/parsedcsv/hjbills_raw.csv', index=False)
vote_df.to_csv('../../data/parsedcsv/hjrollcallvotes_raw.csv', index=False)

Read the files from CSV to test that they wrote properly

In [353]:
bill_df_from_csv = pd.read_csv('../../data/parsedcsv/hjbills_raw.csv')

bill_df_from_csv.head()

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count,validated
0,183,QUORUM,01/08/2003,01:44 PM,1,151,0.0,9.0,True
1,183,H2003 On motion to Postpone,01/08/2003,02:01 PM,2,45,107.0,8.0,True
2,183,H2003 On Amendment,01/08/2003,02:20 PM,3,41,110.0,9.0,True
3,183,H2003 On Amendment,01/08/2003,02:45 PM,4,44,107.0,9.0,True
4,183,H2003 On Amendment,01/08/2003,04:03 PM,5,28,119.0,13.0,True


In [354]:
vote_df_from_csv = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_raw.csv')
vote_df_from_csv.head()

,court_number,rollcall_count_for_year,voter_name,vote_value
0,183,1,Mr. Speaker,Y
1,183,1,Fresolo,Y
2,183,1,"Murphy, K.J.",Y
3,183,1,Swan,Y
4,183,1,Asselin,Y


## Debugging individual PDFs

The following section is used for debugging specific documents that did not work. It is functionally identical to the section ### Testing on a single document, but is kept down here for reference for the work process, and less for presentation purposes.

In [276]:
path = '..\..\data\HJRollCalls\\2016\\2016combined_Vote98.pdf'
doc = pymupdf.open(path)

for page in doc:
    print("NEW DOC STARTING \n\n")
    text = page.get_text()
    lines = text.split('\n')

    for line in lines:
        print(line)


NEW DOC STARTING 


  
 Y    Mr. Speaker     Y    Farley-Bouvier  Y    McMurtry        Y    Velis         
 Y    Mariano         Y    Fernandes       Y    Miceli          Y    Vincent       
 Y    Haddad          Y    Ferrante        Y    Michlewitz      Y    Wagner        
 Y    Rushing         Y    Finn            Y    Mom             Y    Walsh, C.     
 Y    Bradley         Y    Fiola           Y    Moran, F.       Y    Walsh, T.     
 Y    Donato          Y    Fox             Y    Moran, M.       Y    Zlotnik       
 Y    Arciero         Y    Galvin          Y    Murphy          Y    Jones         
 Y    Ashe            Y    Garballey       Y    Nangle          Y    Hill          
 Y    Atkins          Y    Garlick         Y    Naughton        Y    Poirier       
 Y    Ayers           Y    Garry           Y    O'Day           Y    Gifford       
 Y    Balser          Y    Gentile         Y    Parisella       Y    Frost         
 Y    Barber          Y    Golden          Y    Peake

In [166]:
bill_data, vote_data, bill_id = parse_2009through2017_PDF(path, 0, 185, verbose=True)

f0: YEAS: 147         NAYS: 0
f1: N-V: 12
f2: P=PRESENT        X=NOT VOTING      *=AFTER VOTE
f3: Yea and Nay Supplement
f4: JOURNAL OF THE HOUSE.
f5: Thursday, April 2, 2009.
f6: Yea and Nay No. 46
f7: Quorum roll call.
f8: y
f9: Mr. Speaker
f10: y
Dirty name: Orisco11
Clean name: Orisco
f11: Orisco11
f12: y
Dirty name: Linsky
Clean name: Linsky
f13: Linsky
f14: y
Dirty name: Speliotis
Clean name: Speliotis
f15: Speliotis
f16: y
Dirty name: Vallee
Clean name: Vallee
f17: Vallee
f18: y
Dirty name: Dwyer
Clean name: Dwyer
f19: Dwyer
f20: y
Dirty name: L 1 Italien
Clean name: L'Italien
f21: L 1 Italien
f22: y
Dirty name: Spellane
Clean name: Spellane
f23: Spellane
f24: y
Dirty name: Mariano
Clean name: Mariano
f25: Mariano
f26: y
Dirty name: Dykema
Clean name: Dykema
f27: Dykema
f28: X
Dirty name: Madden
Clean name: Madden
f29: Madden
f30: y
Dirty name: Speranza
Clean name: Speranza
f31: Speranza
f32: y
Dirty name: Haddad
Clean name: Haddad
f33: Haddad
f34: y
Dirty name: Ehrlich
Clean na

In [150]:
bill_data

[[0,
  185,
  'On acceptance, in concurrence, of a report of the committee of conference on the disagreeing votes of the two branches (for report see Senate document numbered 2119) with reference to the House amendment to the Senate Bill transferring county sheriffs to the Commonwealth (Senate, No. 2045).',
  '07/29/2009',
  None,
  207,
  152,
  2,
  6,
  False]]

In [121]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count', 'validated'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count,validated
bill_id,,,,,,,,,
0,185,On adoption of an amendment (Acute care hospit...,04/29/2009,None,101,159,0,0,False


In [164]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['court_number', 'rollcall_count_for_year', 'voter_name', 'vote_value'])
vote_df
vote_df.to_csv('temp.csv')

## Remaining TODO items

As of 11/17/2025, there are 2 remaining TODO items to come back to in this file. They are not required for completion of the project, as they relate to older files and have been manually taken care of. However, they are issues that are worth looking at since they may come up in future files.

1) Handling file '..\..\data\HJRollCalls\2016\2016combined_Vote98.pdf' - This file is randomly in a different font than the rest so the data is output in a unique way. It is the only of the 5000 or so roll call votes to appear this way, so it is not worth at the moment writing a new parser for this one special exception but is worth noting the existence of this anomaly. It could be folded into an existing parser since it comes out of PyMuPDF very cleanly with no encoding errors so it should work fine as long as line order is handled correctly but for now it was manually transcribed so the project could progress to completion.
2) Handling the long name with * edge case - In 2 instances in 2017 a longer name (both cases it was a last name joined by a hyphen -) that also had the * indicating an after vote. Both of these instances, the length of the name was enough that the PDF text encoding though there was only a space separating it and the vote of the next rep to the right of the long name representative, so the vote was eaten by the long name and the following rep did not appear in the output. It is an easy to fix case manually especially since it only occurred twice but worth fixing because this may happen again (though it is possible it only was an issue here due to the OCR processing of these specific PDFs).